# 03 — Train an XGBoost model & log evaluation to MLflow

**Input:** `ctl_training_dev.sudsuay_titanic_prep`
**Output:** an MLflow experiment holding one parent run per sweep, nested child runs per
hyperparameter combination, and the best model logged with its signature.

The notebook prefers `xgboost.spark.SparkXGBClassifier` so training stays distributed. If
that isn't importable it falls back to single-node `xgboost.XGBClassifier` — the dataset is
891 rows, so either is fine, and the metrics are computed identically in both cases.

In [ ]:
import json, tempfile, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")             # notebooks still render; nothing needs an interactive backend
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler

import mlflow
import xgboost as xgb
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, log_loss,
                             confusion_matrix, roc_curve, precision_recall_curve)

spark = SparkSession.builder.appName("titanic_train").getOrCreate()

try:
    from xgboost.spark import SparkXGBClassifier
    USE_SPARK_XGB = True
except Exception as exc:            # xgboost < 1.7, or no pyspark integration installed
    USE_SPARK_XGB = False
    print(f"[info] xgboost.spark unavailable ({type(exc).__name__}); using single-node XGBoost")

print(f"xgboost {xgb.__version__} | mlflow {mlflow.__version__} | "
      f"spark backend: {USE_SPARK_XGB}")

## 1. Configuration

In [ ]:
CATALOG      = None
SCHEMA       = "ctl_training_dev"
SOURCE_TABLE = "sudsuay_titanic_prep"
SOURCE_FQN   = ".".join(x for x in [CATALOG, SCHEMA, SOURCE_TABLE] if x)

EXPERIMENT_NAME = "/Shared/sudsuay_titanic_xgb"   # Databricks workspace path
FALLBACK_EXPERIMENT = "sudsuay_titanic_xgb"       # used when not on Databricks
REGISTERED_MODEL = None      # e.g. "ctl_training_dev.sudsuay_titanic_xgb" to register the best run

LABEL_COL = "label"
SEED      = 42

In [ ]:
def setup_experiment(primary, fallback):
    for name in (primary, fallback):
        try:
            mlflow.set_experiment(name)
            print(f"MLflow experiment: {name}")
            print(f"tracking uri      : {mlflow.get_tracking_uri()}")
            return name
        except Exception as exc:
            print(f"[info] could not set experiment {name!r} ({type(exc).__name__}); trying next")
    raise RuntimeError("no usable MLflow experiment")


EXPERIMENT = setup_experiment(EXPERIMENT_NAME, FALLBACK_EXPERIMENT)

## 2. Load the prepared data

`data_split` was frozen in notebook 02, so the holdout is identical on every re-run and
across every experiment in this project. `holdout` rows are the unlabelled Kaggle half —
they are never scored here.

In [ ]:
prep = spark.table(SOURCE_FQN).filter("is_labelled")

FEATURE_COLS = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked",
                "FamilySize", "IsAlone", "SmallFamily", "LogFare", "FarePerPerson",
                "IsChild", "AgeBand", "FareBand", "ClassSex"]

assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features",
                            handleInvalid="error")

train_sdf = assembler.transform(prep.filter("data_split = 'train'")).select("features", LABEL_COL)
test_sdf  = assembler.transform(prep.filter("data_split = 'test'")).select("features", LABEL_COL)

n_train, n_test = train_sdf.count(), test_sdf.count()
pos_rate = train_sdf.agg(F.avg(LABEL_COL)).first()[0]

print(f"train: {n_train:,} rows | test: {n_test:,} rows | {len(FEATURE_COLS)} features")
print(f"train positive rate: {pos_rate:.3f}")

# mild class imbalance -> weight the positive class rather than resample
SCALE_POS_WEIGHT = round((1 - pos_rate) / pos_rate, 3)
print(f"scale_pos_weight   : {SCALE_POS_WEIGHT}")

In [ ]:
# pandas copies for the single-node path and for metric computation
train_pdf = prep.filter("data_split = 'train'").select(*FEATURE_COLS, LABEL_COL).toPandas()
test_pdf  = prep.filter("data_split = 'test'").select(*FEATURE_COLS, LABEL_COL).toPandas()

X_train, y_train = train_pdf[FEATURE_COLS], train_pdf[LABEL_COL]
X_test,  y_test  = test_pdf[FEATURE_COLS],  test_pdf[LABEL_COL]
X_train.shape, X_test.shape

## 3. Train and evaluate helpers

One evaluator for both backends: each training path returns `(model, y_true, y_prob)`, and
everything downstream — metrics, plots, MLflow payloads — works off that.

In [ ]:
def train_model(params, use_spark=USE_SPARK_XGB):
    """Fit XGBoost and return (model, y_true, y_prob) on the held-out test split."""
    if use_spark:
        clf = SparkXGBClassifier(
            features_col="features", label_col=LABEL_COL,
            prediction_col="prediction", probability_col="probability",
            num_workers=1, missing=float("nan"), **params,
        )
        model = clf.fit(train_sdf)
        scored = model.transform(test_sdf).select(LABEL_COL, "probability").toPandas()
        y_true = scored[LABEL_COL].to_numpy()
        y_prob = np.array([float(v[1]) for v in scored["probability"]])
    else:
        model = xgb.XGBClassifier(
            eval_metric="logloss", random_state=SEED, n_jobs=-1, **params
        )
        model.fit(X_train, y_train)
        y_true = y_test.to_numpy()
        y_prob = model.predict_proba(X_test)[:, 1]
    return model, y_true, y_prob


def evaluate(y_true, y_prob, threshold=0.5):
    """Every metric we care about, in one dict ready for mlflow.log_metrics."""
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "roc_auc":   roc_auc_score(y_true, y_prob),
        "pr_auc":    average_precision_score(y_true, y_prob),
        "log_loss":  log_loss(y_true, y_prob, labels=[0, 1]),
        "true_neg": int(tn), "false_pos": int(fp),
        "false_neg": int(fn), "true_pos": int(tp),
    }

In [ ]:
def diagnostic_figure(y_true, y_prob, title):
    """Confusion matrix + ROC + PR curve as a single artifact."""
    y_pred = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4))

    ax = axes[0]
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], ["pred died", "pred survived"])
    ax.set_yticks([0, 1], ["died", "survived"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=14,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_title("Confusion matrix"); ax.grid(False)

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[1].plot(fpr, tpr, color="#2E8B7A", lw=2,
                 label=f"AUC = {roc_auc_score(y_true, y_prob):.3f}")
    axes[1].plot([0, 1], [0, 1], ls="--", lw=1, color="#999")
    axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
    axes[1].set_title("ROC curve"); axes[1].legend(loc="lower right")

    pr, rc, _ = precision_recall_curve(y_true, y_prob)
    axes[2].plot(rc, pr, color="#B4413C", lw=2,
                 label=f"AP = {average_precision_score(y_true, y_prob):.3f}")
    axes[2].axhline(y_true.mean(), ls="--", lw=1, color="#999", label="baseline")
    axes[2].set_xlabel("recall"); axes[2].set_ylabel("precision")
    axes[2].set_title("Precision–recall curve"); axes[2].legend(loc="lower left")

    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    return fig


def importance_figure(model, use_spark=USE_SPARK_XGB, top_n=16):
    booster = model.get_booster() if use_spark else model.get_booster()
    score = booster.get_score(importance_type="gain")
    # Spark path names features f0..fn positionally; map them back to real names
    named = {FEATURE_COLS[int(k[1:])] if k.startswith("f") and k[1:].isdigit() else k: v
             for k, v in score.items()}
    s = pd.Series(named).sort_values(ascending=True).tail(top_n)

    fig, ax = plt.subplots(figsize=(7, 0.32 * len(s) + 1.2))
    ax.barh(s.index, s.values, color="#2E8B7A")
    ax.set_xlabel("gain"); ax.set_title("Feature importance (gain)")
    plt.tight_layout()
    return fig, s

In [ ]:
def log_model_to_mlflow(model, use_spark=USE_SPARK_XGB, name="model"):
    """log_model signature changed in MLflow 3.x — support both."""
    flavor = mlflow.spark if use_spark else mlflow.xgboost
    try:
        return flavor.log_model(model, name=name)                  # MLflow >= 3.0
    except TypeError:
        return flavor.log_model(model, artifact_path=name)         # MLflow 2.x


def log_figure(fig, filename):
    try:
        mlflow.log_figure(fig, filename)
    finally:
        plt.close(fig)

## 4. Baseline run

Sensible defaults, shallow trees, positive class weighted. This is the number every later
run has to beat.

In [ ]:
BASELINE_PARAMS = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.08,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "min_child_weight": 1.0,
    "reg_lambda": 1.0,
    "scale_pos_weight": SCALE_POS_WEIGHT,
}

with mlflow.start_run(run_name="xgb_baseline") as run:
    mlflow.set_tags({
        "stage": "baseline",
        "backend": "spark_xgboost" if USE_SPARK_XGB else "single_node_xgboost",
        "source_table": SOURCE_FQN,
        "dataset": "titanic (labelled half only)",
    })
    mlflow.log_params(BASELINE_PARAMS)
    mlflow.log_params({"n_train": n_train, "n_test": n_test,
                       "n_features": len(FEATURE_COLS), "seed": SEED})

    model, y_true, y_prob = train_model(BASELINE_PARAMS)
    metrics = evaluate(y_true, y_prob)
    mlflow.log_metrics(metrics)

    log_figure(diagnostic_figure(y_true, y_prob, "XGBoost baseline — test split"),
               "diagnostics.png")
    fig_imp, imp = importance_figure(model)
    log_figure(fig_imp, "feature_importance.png")

    mlflow.log_dict({"feature_cols": FEATURE_COLS,
                     "importance_gain": imp.sort_values(ascending=False).round(3).to_dict()},
                    "features.json")
    log_model_to_mlflow(model)

    baseline_run_id = run.info.run_id

print(f"run_id: {baseline_run_id}")
pd.Series(metrics).round(4).to_frame("baseline")

In [ ]:
fig = diagnostic_figure(y_true, y_prob, "XGBoost baseline — test split")
plt.show()
fig_imp, imp = importance_figure(model)
plt.show()

## 5. Hyperparameter sweep

A small explicit grid, each combination as a **nested run** under one parent. Nested runs
keep the experiment readable: the parent carries the winner, the children carry the search.

In [ ]:
from itertools import product

GRID = {
    "max_depth":     [3, 4, 6],
    "learning_rate": [0.05, 0.1],
    "n_estimators":  [200, 400],
}

combos = [dict(zip(GRID, vals)) for vals in product(*GRID.values())]
print(f"{len(combos)} combinations to evaluate")

In [ ]:
PRIMARY_METRIC = "roc_auc"
results = []

with mlflow.start_run(run_name="xgb_grid_search") as parent:
    mlflow.set_tags({"stage": "sweep", "primary_metric": PRIMARY_METRIC,
                     "backend": "spark_xgboost" if USE_SPARK_XGB else "single_node_xgboost"})
    mlflow.log_param("grid", json.dumps(GRID))

    best = {"score": -np.inf}
    for i, combo in enumerate(combos, 1):
        params = {**BASELINE_PARAMS, **combo}
        run_name = "_".join(f"{k}{v}" for k, v in combo.items())

        with mlflow.start_run(run_name=run_name, nested=True) as child:
            mlflow.log_params(params)
            m, yt, yp = train_model(params)
            mets = evaluate(yt, yp)
            mlflow.log_metrics(mets)

            results.append({**combo, **{k: round(v, 4) for k, v in mets.items()},
                            "run_id": child.info.run_id})

            if mets[PRIMARY_METRIC] > best["score"]:
                best = {"score": mets[PRIMARY_METRIC], "params": params, "metrics": mets,
                        "model": m, "y_true": yt, "y_prob": yp,
                        "run_id": child.info.run_id, "name": run_name}

        print(f"  [{i:>2}/{len(combos)}] {run_name:<34} "
              f"{PRIMARY_METRIC}={mets[PRIMARY_METRIC]:.4f}  f1={mets['f1']:.4f}")

    # promote the winner onto the parent run so it is what you see in the UI
    mlflow.log_params({f"best_{k}": v for k, v in best["params"].items()})
    mlflow.log_metrics({f"best_{k}": v for k, v in best["metrics"].items()})
    mlflow.set_tag("best_child_run_id", best["run_id"])

    log_figure(diagnostic_figure(best["y_true"], best["y_prob"],
                                 f"Best model — {best['name']}"), "diagnostics_best.png")
    fig_imp_b, imp_b = importance_figure(best["model"])
    log_figure(fig_imp_b, "feature_importance_best.png")

    leaderboard = (pd.DataFrame(results)
                     .sort_values(PRIMARY_METRIC, ascending=False)
                     .reset_index(drop=True))
    mlflow.log_text(leaderboard.to_csv(index=False), "leaderboard.csv")

    model_info = log_model_to_mlflow(best["model"])
    parent_run_id = parent.info.run_id

print(f"\nbest: {best['name']}  {PRIMARY_METRIC}={best['score']:.4f}")
print(f"parent run: {parent_run_id}")

## 6. Results

In [ ]:
leaderboard[["max_depth", "learning_rate", "n_estimators",
             "roc_auc", "pr_auc", "accuracy", "f1", "precision", "recall", "log_loss"]].head(12)

In [ ]:
comparison = pd.DataFrame({
    "baseline": pd.Series(metrics),
    "best_sweep": pd.Series(best["metrics"]),
})
comparison["delta"] = comparison["best_sweep"] - comparison["baseline"]
comparison.round(4)

In [ ]:
fig = diagnostic_figure(best["y_true"], best["y_prob"], f"Best model — {best['name']}")
plt.show()

## 7. Optionally register the best model

Set `REGISTERED_MODEL` at the top of the notebook to promote the winning run into the
model registry (Unity Catalog on Databricks, or the workspace registry).

In [ ]:
if REGISTERED_MODEL:
    try:
        mlflow.set_registry_uri("databricks-uc")   # no-op / harmless outside Databricks UC
    except Exception:
        pass
    mv = mlflow.register_model(f"runs:/{parent_run_id}/model", REGISTERED_MODEL)
    print(f"registered {REGISTERED_MODEL} version {mv.version}")
else:
    print("REGISTERED_MODEL is None — skipping registry step")

In [ ]:
# Where everything landed
print(f"experiment : {EXPERIMENT}")
print(f"baseline   : {baseline_run_id}")
print(f"sweep      : {parent_run_id}  ({len(combos)} nested runs)")
print("\nlogged per run: params, 11 metrics, diagnostics.png, feature_importance.png, "
      "features.json, the fitted model")

---

### What is in MLflow

| Logged | Detail |
|---|---|
| Params | all XGBoost hyperparameters, split sizes, feature count, seed |
| Metrics | `accuracy`, `precision`, `recall`, `f1`, `roc_auc`, `pr_auc`, `log_loss`, plus raw confusion-matrix counts |
| Artifacts | `diagnostics.png` (confusion matrix + ROC + PR), `feature_importance.png`, `features.json`, `leaderboard.csv` |
| Model | logged with the flavour matching the backend, ready for `mlflow.pyfunc.load_model` |

### Reading the numbers

`roc_auc` is the headline metric here rather than accuracy — the classes are roughly 62/38,
so a model that always predicts "died" scores 62% accuracy while being useless. Watch
`recall` too: false negatives (predicting death for someone who survived) and false
positives are not symmetric costs in most real applications of this pattern.

Expect the sweep to move AUC only slightly. On 891 rows with strong features, XGBoost is
near its ceiling almost immediately — the honest conclusion is usually that the baseline is
good enough and further gains need better features, not better hyperparameters.